# Exploração da API de deputados

Este notebook compara a lista atual de deputados com a lista filtrada pela legislatura 57 e conta os deputados por UF. Os resultados ficam apenas na memória do notebook.

In [10]:
from time import sleep
from typing import Any

import pandas as pd
import requests

API_URL = "https://dadosabertos.camara.leg.br/api/v2/deputados"
REQUEST_TIMEOUT = 30
PAGE_SIZE = 100
REQUEST_DELAY_SECONDS = 0.2
MAX_ATTEMPTS = 3

## Funções para consultar todas as páginas

A primeira página usa os parâmetros da consulta. As páginas seguintes usam o URL `next` informado pela própria API. Cada tentativa tem timeout e no máximo três tentativas; entre páginas há uma pequena pausa para respeitar os limites da API.

In [11]:
def get_json(url: str, params: dict[str, int] | None = None) -> dict[str, Any]:
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == MAX_ATTEMPTS:
                raise
            sleep(REQUEST_DELAY_SECONDS * attempt)
    raise RuntimeError("Request attempts finished unexpectedly.")


def fetch_deputies(legislature_id: int | None = None) -> list[dict[str, Any]]:
    params = {"itens": PAGE_SIZE, "ordem": "ASC", "ordenarPor": "nome"}
    if legislature_id is not None:
        params["idLegislatura"] = legislature_id

    deputies: list[dict[str, Any]] = []
    next_url: str | None = API_URL
    first_page = True

    while next_url is not None:
        page = get_json(next_url, params=params if first_page else None)
        deputies.extend(page["dados"])
        next_links = [link["href"] for link in page.get("links", []) if link.get("rel") == "next"]
        next_url = next_links[0] if next_links else None
        first_page = False
        if next_url is not None:
            sleep(REQUEST_DELAY_SECONDS)

    return deputies

## Buscar as duas listas

A primeira consulta não envia `idLegislatura`; a segunda envia `idLegislatura=57`.

In [3]:
deputies_all = fetch_deputies()
deputies_legislature_57 = fetch_deputies(legislature_id=57)

print(f"Total without legislature filter: {len(deputies_all)}")
print(f"Total in legislature 57: {len(deputies_legislature_57)}")

Total without legislature filter: 513
Total in legislature 57: 877


## Comparar ids

As diferenças abaixo mostram quais ids existem somente em cada resultado.

In [4]:
ids_all = {deputy["id"] for deputy in deputies_all}
ids_legislature_57 = {deputy["id"] for deputy in deputies_legislature_57}

only_without_filter = sorted(ids_all - ids_legislature_57)
only_in_legislature_57 = sorted(ids_legislature_57 - ids_all)

print("Ids only in the unfiltered list:", only_without_filter)
print("Ids only in legislature 57:", only_in_legislature_57)

Ids only in the unfiltered list: []
Ids only in legislature 57: [68720, 73531, 73806, 74052, 74065, 74095, 74140, 74141, 74159, 74283, 74295, 74299, 74383, 74696, 77003, 89536, 92346, 100689, 103758, 115200, 116379, 122195, 128760, 129618, 132504, 138286, 141480, 143942, 154700, 160531, 163321, 164360, 171139, 178909, 182146, 194260, 204376, 204422, 204433, 204448, 204476, 204503, 204507, 204514, 204522, 204558, 204573, 205865, 208297, 212625, 213274, 217480, 218086, 219585, 220008, 220531, 220575, 220577, 220579, 220580, 220587, 220593, 220596, 220600, 220601, 220619, 220627, 220634, 220636, 220639, 220659, 220705, 221339, 221378, 222142, 222429, 222565, 223128, 223398, 225386, 225727, 225730, 226179, 226410, 226553, 226708, 227275, 227310, 227323, 227324, 227370, 227401, 227433, 227660, 227800, 228042, 228272, 228616, 228797, 228837, 228941, 229069, 229082, 229106, 229112, 229225, 229259, 229333, 229432, 229585, 229939, 230088, 230177, 230762, 230764, 231911, 233218, 233598, 234253, 

## Deputados por UF

A contagem usa a lista sem filtro de legislatura e o campo bruto `siglaUf` retornado pela API.

In [12]:
deputies_df = pd.DataFrame(deputies_all)
deputies_by_state = (
    deputies_df.groupby("siglaUf", dropna=False)["id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("deputy_count")
    .reset_index()
)
deputies_by_state

,siglaUf,deputy_count
0,SP,70
1,MG,53
2,RJ,46
3,BA,39
4,RS,31
5,PR,30
6,PE,25
7,CE,22
8,MA,18
9,PA,17


## Despesas por deputado em 2025

A função segue o link next e anexa idDeputado a cada registro. O documento do fornecedor é mantido como texto.

In [6]:
from decimal import Decimal, InvalidOperation

EXPENSES_URL = "https://dadosabertos.camara.leg.br/api/v2/deputados/{deputy_id}/despesas"

def fetch_expenses(deputy_id: int, year: int) -> tuple[list[dict[str, Any]], int]:
    url = EXPENSES_URL.format(deputy_id=deputy_id)
    params = {"ano": year, "itens": PAGE_SIZE}
    records: list[dict[str, Any]] = []
    page_count = 0
    first_page = True
    while url is not None:
        page = get_json(url, params=params if first_page else None)
        page_count += 1
        records.extend({**row, "idDeputado": deputy_id} for row in page["dados"])
        next_links = [link["href"] for link in page.get("links", []) if link.get("rel") == "next"]
        url = next_links[0] if next_links else None
        first_page = False
        if url is not None:
            sleep(REQUEST_DELAY_SECONDS)
    return records, page_count

def decimal_value(value: Any) -> Decimal | None:
    if value is None or pd.isna(value):
        return None
    try:
        return Decimal(str(value))
    except (InvalidOperation, ValueError):
        return None

## Duas consultas para 2025

Seleciona-se um deputado da lista sem filtro, que deve conter 513 pessoas, e um id que aparece somente na lista da legislatura 57. São exibidos apenas ids e contagens agregadas.

In [7]:
if len(deputies_all) != 513:
    raise ValueError(f"Expected 513 deputies, got {len(deputies_all)}")
if not only_in_legislature_57:
    raise ValueError("No ids found only in legislature 57")
    
selected_deputy_ids = {"unfiltered_list": sorted(ids_all)[4], "only_in_legislature_57": only_in_legislature_57[7]}
expense_results: dict[str, dict[str, Any]] = {}

for group_name, deputy_id in selected_deputy_ids.items():
    records, pages = fetch_expenses(deputy_id, year=2025)
    expenses_df = pd.DataFrame(records)
    if "cnpjCpfFornecedor" in expenses_df.columns:
        expenses_df["cnpjCpfFornecedor"] = expenses_df["cnpjCpfFornecedor"].astype("string")
    null_counts = expenses_df.isna().sum().rename("null_count").to_frame()
    expense_results[group_name] = {"deputy_id": deputy_id, "pages": pages, "dataframe": expenses_df}
    print(f"{group_name}: deputy id={deputy_id}, pages={pages}, records={len(records)}")
    display(null_counts)

unfiltered_list: deputy id=72442, pages=7, records=661


,null_count
ano,0
mes,0
tipoDespesa,0
codDocumento,0
tipoDocumento,0
codTipoDocumento,0
dataDocumento,9
numDocumento,0
valorDocumento,0
urlDocumento,274


only_in_legislature_57: deputy id=74141, pages=7, records=624


,null_count
ano,0
mes,0
tipoDespesa,0
codDocumento,0
tipoDocumento,0
codTipoDocumento,0
dataDocumento,9
numDocumento,0
valorDocumento,0
urlDocumento,152


In [8]:
negative_expense_columns = [
    "ano",
    "mes",
    "tipoDespesa",
    "tipoDocumento",
    "codTipoDocumento",
    "valorDocumento",
    "valorGlosa",
    "valorLiquido",
]

for group_name, result in expense_results.items():
    expenses_df = result["dataframe"]
    print(f"{group_name} (deputy id={result['deputy_id']})")
    if "valorDocumento" not in expenses_df.columns:
        negative_expenses = pd.DataFrame(columns=negative_expense_columns)
    else:
        negative_expenses = expenses_df.loc[
            expenses_df["valorDocumento"] < 0, negative_expense_columns
        ]
    display(negative_expenses)


unfiltered_list (deputy id=72442)


,ano,mes,tipoDespesa,tipoDocumento,codTipoDocumento,valorDocumento,valorGlosa,valorLiquido
28,2025,1,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1469.95,0.0,-1469.95
42,2025,1,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-554.65,0.0,-554.65
52,2025,1,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-2191.92,0.0,-2191.92
93,2025,2,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1469.95,0.0,-1469.95
98,2025,2,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1878.28,0.0,-1878.28
100,2025,2,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1829.78,0.0,-1829.78
145,2025,3,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-751.30,0.0,-751.30
205,2025,4,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1413.13,0.0,-1413.13
206,2025,4,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1523.52,0.0,-1523.52
212,2025,4,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-844.91,0.0,-844.91


only_in_legislature_57 (deputy id=74141)


,ano,mes,tipoDespesa,tipoDocumento,codTipoDocumento,valorDocumento,valorGlosa,valorLiquido
51,2025,1,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-256.80,0.0,-256.80
118,2025,2,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-365.90,0.0,-365.90
173,2025,3,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1944.95,0.0,-1944.95
184,2025,3,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-394.80,0.0,-394.80
290,2025,5,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-103.68,0.0,-103.68
291,2025,5,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-103.68,0.0,-103.68
292,2025,5,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-158.25,0.0,-158.25
381,2025,7,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-30.95,0.0,-30.95
390,2025,7,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-30.95,0.0,-30.95
391,2025,7,PASSAGEM AÉREA - SIGEPA,Nota Fiscal,0,-1876.78,0.0,-1876.78


## Verificação de campos de valor
Visa encontrar se valorLiquido ou valorDocumento - valorGlosa bate com o portal da transparência

In [13]:
for group_name, result in expense_results.items():
    expenses_df = result["dataframe"]
    document_values = expenses_df["valorDocumento"].map(decimal_value)
    deduction_values = expenses_df["valorGlosa"].map(decimal_value)
    net_values = expenses_df["valorLiquido"].map(decimal_value)

    document_minus_deduction_total = sum(
        (document - deduction
         for document, deduction in zip(document_values, deduction_values)
         if document is not None and deduction is not None),
        Decimal("0"),
    )
    net_total = sum(
        (net for net in net_values if net is not None),
        Decimal("0"),
    )

    print(f"{group_name} (deputy id={result['deputy_id']})")
    print(f"Sum of valorDocumento - valorGlosa: {document_minus_deduction_total}")
    print(f"Sum of valorLiquido: {net_total}")

unfiltered_list (deputy id=72442)
Sum of valorDocumento - valorGlosa: 565373.53
Sum of valorLiquido: -168944.19
only_in_legislature_57 (deputy id=74141)
Sum of valorDocumento - valorGlosa: 530023.56
Sum of valorLiquido: -310906.58


## Verificações monetárias

Os valores são comparados como Decimal para evitar imprecisão binária. Linhas sem algum dos valores necessários são contadas separadamente.

In [9]:
for group_name, result in expense_results.items():
    expenses_df = result["dataframe"]
    required = {"valorLiquido", "valorDocumento", "valorGlosa"}
    missing = required - set(expenses_df.columns)
    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")
    amounts = expenses_df[["valorLiquido", "valorDocumento", "valorGlosa"]].apply(lambda col: col.map(decimal_value))
    comparable = amounts.notna().all(axis=1)
    matches = amounts.loc[comparable, "valorLiquido"].abs() == (amounts.loc[comparable, "valorDocumento"] - amounts.loc[comparable, "valorGlosa"])
    negative_documents = int((amounts["valorDocumento"].dropna() < Decimal("0")).sum())
    print(f"{group_name} (deputy id={result["deputy_id"]}): checked={int(comparable.sum())}, matches={int(matches.sum())}, mismatches={int((~matches).sum())}, missing_amounts={int((~comparable).sum())}, negative_valorDocumento={negative_documents}")

unfiltered_list (deputy id=72442): checked=661, matches=618, mismatches=43, missing_amounts=0, negative_valorDocumento=43
only_in_legislature_57 (deputy id=74141): checked=624, matches=608, mismatches=16, missing_amounts=0, negative_valorDocumento=16
